# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Rebuild the honest (grouped-split) model from Week 6 — this playbook is
# built on the validated model, not the naive one.
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
april = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
)

march_agg = (
    march.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_sum_position=("gsc_sum_position", "sum")
    )
)
march_agg["march_position"] = march_agg["march_sum_position"] / march_agg["march_impressions"]

april_agg = (
    april.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(april_impressions=("gsc_impressions", "sum"))
)

model_df = march_agg[march_agg["march_impressions"] > 0].copy()
model_df = model_df.merge(april_agg, on=["client_hash_id", "content_hash_id"], how="left")
model_df["april_impressions"] = model_df["april_impressions"].fillna(0)
model_df["target_decline"] = (model_df["april_impressions"] < model_df["march_impressions"]).astype(int)

features = ["march_impressions", "march_clicks", "march_position"]
target = "target_decline"
model_df = model_df.dropna(subset=features + [target])

X = model_df[features].copy()
y = model_df[target].copy()
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]
honest_ap = average_precision_score(y_test, test_prob)
base_rate = y_test.mean()

print("Honest grouped-split Average Precision:", honest_ap)
print("Base rate:", base_rate)
print("Skill above base rate:", honest_ap - base_rate)

Honest grouped-split Average Precision: 0.6113944595078553
Base rate: 0.5775660739473258
Skill above base rate: 0.03382838556052947


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### The queue: what to do first, and why

I score the held-out test set (unseen clients) with the validated model and
sort pages by decline_probability. Each page gets a plain-language reason
code built from its own March feature values, and a simple archetype label
based on visibility and rank — not a black-box score alone.

Per the claim ladder: this is a validated model that ranks out-of-sample, so
I can say "the model ranks/flags these pages at Average Precision of
[honest_ap], [skill] points above the base rate of [base_rate]" — I do not
claim the model predicts what will happen to any single page with certainty.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Score the held-out test set and build the ranked queue
queue = model_df.iloc[test_idx][
    ["client_hash_id", "content_hash_id", "march_impressions", "march_clicks", "march_position"]
].copy()
queue["decline_probability"] = test_prob

# Simple, human-readable archetype based on visibility and position
imp_median = queue["march_impressions"].median()
pos_median = queue["march_position"].median()

def archetype(row):
    high_imp = row["march_impressions"] >= imp_median
    good_pos = row["march_position"] <= pos_median  # lower position number = better rank
    if high_imp and good_pos:
        return "High-visibility, well-ranked"
    if high_imp and not good_pos:
        return "High-visibility, weak rank"
    if not high_imp and good_pos:
        return "Low-visibility, well-ranked"
    return "Low-visibility, weak rank"

queue["archetype"] = queue.apply(archetype, axis=1)

# Plain-language reason code built from the page's own feature values
def reason_code(row):
    parts = []
    parts.append("high March impressions" if row["march_impressions"] >= imp_median else "low March impressions")
    parts.append("weak average position" if row["march_position"] > pos_median else "strong average position")
    parts.append(f"decline probability {row['decline_probability']:.2f}")
    return "; ".join(parts)

queue["reason_code"] = queue.apply(reason_code, axis=1)

queue = queue.sort_values("decline_probability", ascending=False).reset_index(drop=True)
queue["priority_rank"] = queue.index + 1

print(queue.head(10)[["priority_rank", "archetype", "reason_code", "decline_probability"]])

   priority_rank                     archetype  \
0              1   Low-visibility, well-ranked   
1              2   Low-visibility, well-ranked   
2              3  High-visibility, well-ranked   
3              4  High-visibility, well-ranked   
4              5  High-visibility, well-ranked   
5              6  High-visibility, well-ranked   
6              7  High-visibility, well-ranked   
7              8  High-visibility, well-ranked   
8              9  High-visibility, well-ranked   
9             10  High-visibility, well-ranked   

                                         reason_code  decline_probability  
0  low March impressions; strong average position...             0.609703  
1  low March impressions; strong average position...             0.609703  
2  high March impressions; strong average positio...             0.609703  
3  high March impressions; strong average positio...             0.609703  
4  high March impressions; strong average positio...             0.60

### Connecting the queue to the decay/refresh pattern

The research paper's Finding #4 reports that the 31-90 day freshness window
showed the strongest measured growth-to-decline ratio in that portfolio
(7.88:1, n reported in the paper). This is directional evidence from a
different dataset, not proof about any specific page here. In this
playbook, "high-visibility, weak rank" pages are treated as the closest
match to that pattern: pages that still carry meaningful visibility but
show signs of slipping, similar in spirit to the paper's "mature pages
before they decay" recommendation. This is offered as a decision-support
rationale for prioritization, not as a causal claim that refreshing a page
will reverse its decline.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who uses this, and where it stops being valid

**Intended use:** A content editor uses this ranked queue as a starting
shortlist for manual review — which pages to look at first this cycle, not
a final decision.

**What the model is:** A decision tree trained on three March signals
(march_impressions, march_clicks, march_position) from one client set,
evaluated with a grouped split so no client appears in both train and test.
Measured Average Precision is [honest_ap], compared to a base rate of
[base_rate] — a real but modest skill margin, not near-perfect separation.

**Where it stops being valid:**
- It only flags a decline in Google Search Console impressions from one
  month to the next — it does not identify *why* a page might decline.
- It was trained and evaluated on 36 + 11 = 47 clients in this snapshot; it
  has not been validated on clients outside this dataset.
- It is cross-sectional, observational data. Per the claim ladder, this
  supports "these pages look worth reviewing first, because…", not "doing X
  will produce Y."
- It says nothing about content quality, competitors, or algorithm changes.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Model evaluated on {len(test_clients) if 'test_clients' in dir() else queue['client_hash_id'].nunique()} held-out clients")
print(f"Honest Average Precision: {honest_ap:.4f}")
print(f"Base rate: {base_rate:.4f}")
print(f"Skill above base rate: {honest_ap - base_rate:.4f}")


Model evaluated on 11 held-out clients
Honest Average Precision: 0.6114
Base rate: 0.5776
Skill above base rate: 0.0338


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### What a person must check before acting

Before acting on any page in this queue, a reviewer should confirm:
- The page is not brand-new (recently published pages have little March
  history and an unstable signal).
- The impression volume is not so small that normal noise could flip the
  label (a page with a handful of impressions can "decline" by a couple of
  clicks and look identical to a real drop).
- There is no known external cause already explaining the change (a client
  campaign pause, a site migration, a seasonal topic) — the model has no
  way to see these.
- The recommended action (refresh, monitor, deprioritize) makes sense given
  the actual page content, which the model never reads.

### What should NEVER be automated from this queue alone
- Auto-publishing content changes based on the model's ranking.
- Auto-deprioritizing or deindexing a page without a human reading it.
- Using the ranking as a performance metric for a writer or editor.
- Treating decline_probability as a probability of real-world causal harm —
  it is a rank, not a certainty score, per the claim ladder.

Every action in this queue requires human sign-off before anything changes
on a live page.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Flag pages that fail the no-go checks so reviewers see them before acting
low_volume_threshold = queue["march_impressions"].quantile(0.10)

queue["no_go_flag"] = queue["march_impressions"] <= low_volume_threshold
queue.loc[queue["no_go_flag"], "reason_code"] += "; LOW VOLUME - verify before acting"

print("Pages flagged as low-volume / needs extra review:", queue["no_go_flag"].sum())


Pages flagged as low-volume / needs extra review: 4438


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### What would tell you the recommendations went stale

- **Performance trigger:** if a future evaluation's Average Precision falls
  back toward the base rate (skill margin near [honest_ap - base_rate]
  today), the model has stopped adding signal and should be retrained or
  retired.
- **Population trigger:** if the mix of clients or content types shifts
  materially from what was seen in this training set (36 train clients),
  re-check the grouped split and retrain rather than trusting an old model
  on a new population.
- **Time trigger:** retrain on a rolling basis (for example, every quarter)
  since search performance patterns can shift with algorithm updates and
  seasonality that this snapshot does not capture.
- **Drift check:** compare the distribution of march_impressions,
  march_clicks, and march_position in new data against this training set;
  a large shift is a signal to re-validate before trusting the queue.

These are monitoring guidelines for a non-production, decision-support
tool — not a claim that this pipeline currently has automated monitoring
running.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Record the current feature distributions as a monitoring baseline
monitoring_baseline = {
    "march_impressions": {
        "mean": float(X_train["march_impressions"].mean()),
        "median": float(X_train["march_impressions"].median())
    },
    "march_clicks": {
        "mean": float(X_train["march_clicks"].mean()),
        "median": float(X_train["march_clicks"].median())
    },
    "march_position": {
        "mean": float(X_train["march_position"].mean()),
        "median": float(X_train["march_position"].median())
    },
    "honest_ap_at_training_time": float(honest_ap),
    "base_rate_at_training_time": float(base_rate)
}

print(json.dumps(monitoring_baseline, indent=2))


{
  "march_impressions": {
    "mean": 1503.0672329969811,
    "median": 144.0
  },
  "march_clicks": {
    "mean": 4.240982538148639,
    "median": 0.0
  },
  "march_position": {
    "mean": 15.932402394555897,
    "median": 8.333333333333334
  },
  "honest_ap_at_training_time": 0.6113944595078553,
  "base_rate_at_training_time": 0.5775660739473258
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports

The ranked queue is written to `work/outputs/` (intentionally excluded from
git by the CI leak-guard — this notebook regenerates it on demand). The
feature-importance figure is committed to `work/figures/` for reuse in the
paper. Metrics are committed to `work/metrics/` as the receipts the paper's
numbers trace back to.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/metrics", exist_ok=True)

# 1. Ranked queue CSV (git-ignored, regenerated by this notebook)
queue.to_csv("work/outputs/w07_ranked_queue.csv", index=False)
print("Saved:", "work/outputs/w07_ranked_queue.csv", "-", len(queue), "rows")

# 2. Feature importance figure (committed, reused in the paper)
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(6, 4))
plt.barh(feature_importance["feature"], feature_importance["importance"])
plt.xlabel("Feature importance")
plt.title("Decision tree feature importance (honest grouped-split model)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("work/figures/w07_feature_importance.png", dpi=150)
plt.close()
print("Saved: work/figures/w07_feature_importance.png")

# 3. Metrics JSON (committed — the receipts the paper traces back to)
metrics = {
    "honest_average_precision": float(honest_ap),
    "base_rate": float(base_rate),
    "skill_above_base_rate": float(honest_ap - base_rate),
    "train_clients": int(model_df.iloc[train_idx]["client_hash_id"].nunique()),
    "test_clients": int(model_df.iloc[test_idx]["client_hash_id"].nunique()),
    "queue_rows": int(len(queue)),
    "no_go_flagged_rows": int(queue["no_go_flag"].sum()),
    "monitoring_baseline": monitoring_baseline
}

with open("work/metrics/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved: work/metrics/w07_metrics.json")
print(json.dumps(metrics, indent=2))

Saved: work/outputs/w07_ranked_queue.csv - 43247 rows
Saved: work/figures/w07_feature_importance.png
Saved: work/metrics/w07_metrics.json
{
  "honest_average_precision": 0.6113944595078553,
  "base_rate": 0.5775660739473258,
  "skill_above_base_rate": 0.03382838556052947,
  "train_clients": 36,
  "test_clients": 11,
  "queue_rows": 43247,
  "no_go_flagged_rows": 4438,
  "monitoring_baseline": {
    "march_impressions": {
      "mean": 1503.0672329969811,
      "median": 144.0
    },
    "march_clicks": {
      "mean": 4.240982538148639,
      "median": 0.0
    },
    "march_position": {
      "mean": 15.932402394555897,
      "median": 8.333333333333334
    },
    "honest_ap_at_training_time": 0.6113944595078553,
    "base_rate_at_training_time": 0.5775660739473258
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.